In [ ]:
import os
import json
import requests
import pandas as pd
import re

from jsonschema import validate, ValidationError
from dotenv import load_dotenv

In [ ]:
load_dotenv()

api_key = os.getenv("LLM_API_KEY")

print("API key loaded:", api_key is not None)

In [ ]:
def call_llm(system_prompt, user_prompt, temperature=0.0, max_tokens=512):

    url = "https://openrouter.ai/api/v1/chat/completions"

    headers = {
        "Authorization": f"Bearer {api_key}",
        "Content-Type": "application/json"
    }

    payload = {

        "model":"openai/gpt-3.5-turbo",

        "messages":[
            {
                "role":"system",
                "content":system_prompt
            },
            {
                "role":"user",
                "content":user_prompt
            }
        ],

        "temperature":temperature,
        "max_tokens":max_tokens
    }


    response = requests.post(
        url,
        headers=headers,
        json=payload
    )


    if response.status_code != 200:
        print(response.status_code)
        return None


    return response.json()["choices"][0]["message"]["content"]

In [ ]:
def has_pii(text):

    email_pattern = r'[a-zA-Z0-9_.+-]+@[a-zA-Z0-9-]+\.[a-zA-Z0-9-.]+'

    phone_pattern = r'\b\d{10}\b'

    return bool(
        re.search(email_pattern,text)
        or
        re.search(phone_pattern,text)
    )


def safe_llm_call(system,user):

    if has_pii(user):
        print("Input blocked: PII detected.")
        return None

    return call_llm(system,user)

In [ ]:
df = pd.read_csv(
"../part2/data/cleaned_data.csv"
)


records = df.head(3)


records

In [ ]:
schema = {

"type":"object",

"properties":{

"risk_tier":{"type":"string"},
"flag_for_review":{"type":"boolean"},
"primary_signal":{"type":"string"},
"confidence":{"type":"string"},
"recommended_action":{"type":"string"}

},

"required":[

"risk_tier",
"flag_for_review",
"primary_signal",
"confidence",
"recommended_action"

]

}

In [ ]:
system_prompt = """

You are a medical risk assessment assistant.

Score patient records.

Return ONLY JSON.

Risk criteria:

High risk:
older age, high glucose, hypertension,
heart disease.

Low risk:
younger age and fewer risk factors.

Example:

Input:
age:70, glucose:200

Output:
{
"risk_tier":"high",
"flag_for_review":true,
"primary_signal":"high glucose",
"confidence":"high",
"recommended_action":"medical review"
}

"""

In [ ]:
results=[]


for i,row in records.iterrows():

    user_prompt = row.to_json()


    output = safe_llm_call(
        system_prompt,
        user_prompt
    )


    try:

        parsed=json.loads(output.strip())

        validate(
            parsed,
            schema
        )

        status="PASS"


    except Exception as e:

        parsed={
        "risk_tier":None,
        "flag_for_review":None,
        "primary_signal":None,
        "confidence":None,
        "recommended_action":None
        }

        status="FAIL"


    results.append(
        [i,parsed,status]
    )


results

In [ ]:
for temp in [0,0.7]:

    output = call_llm(
        system_prompt,
        records.iloc[0].to_json(),
        temperature=temp
    )

    print("Temperature:",temp)
    print(output)

In [ ]:
print(
safe_llm_call(
system_prompt,
"contact me at test@gmail.com"
)
)


print(
safe_llm_call(
system_prompt,
"age 60 glucose 150"
)
)